# 1 加载数据集
## Dataset

In [ ]:
from torch.utils.data import Dataset
import json

class AFQMC(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt') as f:  # 以只读文本模式打开数据文件（read text）
            for idx, line in enumerate(f):  # enumerate()为可迭代对象添加计数器；f：文件对象，可被逐行迭代；返回值：生成器，每次产生一个（索引，元素）的元组
                sample = json.loads(line.strip())  # strip()去除首尾的空白字符和换行符；是loads（从字符串加载），不是load（从文件加载）
                Data[idx] = sample
        return Data
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_data = AFQMC('data/afqmc_public/train.json')

print(train_data[0])

In [13]:
from torch.utils.data import Dataset # 下面的的代码用的是这个数据集
import json

class AFQMC(Dataset):
    def __init__(self, data_file):
        super().__init__()
        self.data = self.load_data(data_file)

    def load_data(self, data_file):
        data_res = {}
        with open(data_file, "rt") as f:
            for index, line in enumerate(f):
                sample = json.loads(line.strip())
                data_res[index] = sample

        return data_res
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return self.data[index]
    
train_data = AFQMC("data/afqmc_public/train.json")

print(train_data.__getitem__(0))

{'sentence1': '蚂蚁借呗等额还款可以换成先息后本吗', 'sentence2': '借呗有先息到期还本吗', 'label': '0'}


In [ ]:
# 如果数据集非常巨大，难以一次性加载到内存中，也可以继承IterableDataset类构建迭代型数据集(不能用shuffle)
from torch.utils.data import IterableDataset
import json

class IterableAFQMC(IterableDataset):
    def __init__(self, data_file):
        self.data_file = data_file  # 没有立即加载数据到内存，只是保存路径

    def __iter__(self):
        with open(self.data_file, 'rt') as f:
            for line in f:      # 延迟加载（lazy loading）的关键：python会逐行读取文件，而不是一次性读取整个文件
                sample = json.loads(line.strip())
                yield sample  # yield关键字将该类变成一个生成器，它会返回当前行的数据，然后暂停执行，直到下一次请求数据时再继续读取下一行

train_data = IterableAFQMC('data/afqmc_public/train.json')

# iter(train_data)调用类中的__iter__方法，创建一个迭代器对象；next()从迭代器中请求第一个元素，此时程序会进入for循环，读取文件的第一行，并执行yield
print(next(iter(train_data)))  

In [10]:
from torch.utils.data import IterableDataset
import json

class IterableAFQMC(IterableDataset):
    def __init__(self, data_file):
        super().__init__()
        self.data_file = data_file

    def __iter__(self):
        with open(self.data_file, "rt") as f:
            for line in f:
                sample = json.loads(line.strip())
                yield sample

train_data = IterableAFQMC("data/afqmc_public/train.json")
# test_data = IterableAFQMC("data/afqmc_public/test.json")  # 这个数据集没有label，不能当valid_data

print(next(iter(train_data)))

{'sentence1': '蚂蚁借呗等额还款可以换成先息后本吗', 'sentence2': '借呗有先息到期还本吗', 'label': '0'}


## DataLoader

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

checkpoint = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def collate_fn(batch_samples):  # collate_fn：DataLoader将多个样本（samples）打包成一个批次(batch)
    batch_sentence_1, batch_sentence_2 = [], []
    batch_label = []
    for sample in batch_samples:  
        batch_sentence_1.append(sample['sentence1'])
        batch_sentence_2.append(sample['sentence2'])
        batch_label.append(int(sample['label']))
    X = tokenizer(
        batch_sentence_1,  # 同时传入2个列表tokenizer会自动处理成BERT需要的双句子输入格式:[CLS]句子A[SEP]句子B[SEP]
        batch_sentence_2, 
        padding=True, 
        truncation=True, 
        return_tensors="pt"  # 是return_tensors，不是return_tensor
    )
    y = torch.tensor(batch_label)
    return X, y

train_dataloader = DataLoader(train_data, batch_size=4, shuffle=True, collate_fn=collate_fn)

batch_X, batch_y = next(iter(train_dataloader))
print('batch_X shape:', {k: v.shape for k, v in batch_X.items()})
print('batch_y shape:', batch_y.shape)
print(batch_X)
print(batch_y)

# tokenizer之后的输出：101是[CLS], 102是[SEP]；[CLS]句子A[SEP]的token_type_id是0，句子B[SEP]的token_type_id是1，后面的token_type_id=0都是填充值（它们的attention_mask都是0）

In [15]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

checkpoint = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def collate_fn(batch_sample):
    batch_sentence_1, batch_sentence_2, labels = [], [], []
    for sample in batch_sample:
        batch_sentence_1.append(sample["sentence1"])
        batch_sentence_2.append(sample["sentence2"])
        labels.append(int(sample["label"]))

    X = tokenizer(batch_sentence_1,
                  batch_sentence_2,
                  padding = True,
                  truncation = True,
                  return_tensors = "pt")
    y = torch.tensor(labels)

    return X,y

train_dataloader = DataLoader(dataset = train_data, batch_size = 4, shuffle = True, collate_fn = collate_fn)
valid_dataloader = DataLoader(dataset = train_data, batch_size = 8, shuffle = False, collate_fn = collate_fn)

batch_X, batch_y = next(iter(train_dataloader))

# print(f"batch_X size: {k: v.size() for k,v in batch_X.items()}")
print("batch_X size:", {k: v.size() for k,v in batch_X.items()})
print(f"batch_y size: {batch_y.size()}")
print(batch_X)
print(batch_y)


batch_X size: {'input_ids': torch.Size([4, 37]), 'token_type_ids': torch.Size([4, 37]), 'attention_mask': torch.Size([4, 37])}
batch_y size: torch.Size([4])
{'input_ids': tensor([[ 101,  955, 1446, 6206, 2130, 1587,  928, 2622,  102, 6010, 6009,  955,
         1446, 6716,  819,  928, 2622, 2130, 1587,  102,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0],
        [ 101, 7312, 7824,  677, 1377,  809, 4500, 5709, 1446, 1658,  102,  743,
          691, 6205, 4500,  679,  749, 5709, 1446,  102,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0],
        [ 101, 5709, 1446,  955, 3621, 3189, 5543,  934, 3121, 1408,  102, 3291,
         2940, 5709, 1446, 6820, 3621, 3189, 3309,  102,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0],
        [ 101, 6010, 6009,  955, 1446, 2769,  955,  

# 2 训练模型
## 构建模型

In [ ]:
from torch import nn
from transformers import AutoModel

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using {device} device')

class BertForPairwiseCLS(nn.Module):
    def __init__(self):
        super(BertForPairwiseCLS, self).__init__()
        self.bert_encoder = AutoModel.from_pretrained(checkpoint)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, 2)

    def forward(self, x):
        bert_output = self.bert_encoder(**x)
        cls_vectors = bert_output.last_hidden_state[:, 0, :]
        cls_vectors = self.dropout(cls_vectors)
        logits = self.classifier(cls_vectors)
        return logits
    
model = BertForPairwiseCLS().to(device)
print(model)

In [ ]:
from torch import nn
from transformers import AutoConfig
# BertPreTrainedModel是关键基类，继承它意味着你的模型可以像官方模型一样使用.pretrained()加载权重，并支持自动保存
# BertModel: BERT的核心层，也就是transformer层，不带最后的分类头
from transformers import BertPreTrainedModel, BertModel 

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using {device} device')

class BertForPairwiseCLS(BertPreTrainedModel):
    def __init__(self, config):  # 接收一个config对象，包含了模型的所有超参数（如 hidden_size, num_hidden_layers 等）
        super().__init__(config)  # 调用基类初始化，确保config会被正确存储在self.config中
        self.bert = BertModel(config, add_pooling_layer=False)  # 把原生的pooler层（提取CLS后再接一个Linear+Tanh）关掉，因为后面要手动提取CLS
        self.dropout = nn.Dropout(config.hidden_dropout_prob) # dropout_rate直接从配置文件中读
        self.classifier = nn.Linear(768, 2)  # BERT Base的输出维度是768
        self.post_init()  # 自动对self.classifier的权重进行初始化，用BERT论文中建议的初始化方案，而不是随机初始化（这样会导致模型收敛困难），这个一定要记得写！！！
    
    def forward(self, x):
        bert_output = self.bert(**x) # 将输入的字典（input_ids, token_type_ids, attension_mask）解包传给bert
        cls_vectors = bert_output.last_hidden_state[:, 0, :]  # 提取出每个batch中第0个位置的向量，也就是[CLS]，记得加last_hidden_state
        cls_vectors = self.dropout(cls_vectors)
        logits = self.classifier(cls_vectors)
        return logits

config = AutoConfig.from_pretrained(checkpoint) # 加载预训练模型对应的配置（知道模型有几层，多宽）
model = BertForPairwiseCLS.from_pretrained(checkpoint, config=config).to(device)
print(model)

In [17]:
from torch import nn
from transformers import AutoConfig, BertModel, BertPreTrainedModel

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using {device} device')

class BertForPairwiseCLS(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config, add_pooling_layer = False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(768, 2)
        self.post_init()

    def forward(self, x):
        bert_output = self.bert(**x)
        cls_vectors = bert_output.last_hidden_state[:,0,:]
        cls_vectors = self.dropout(cls_vectors)
        logits = self.classifier(cls_vectors)
        return logits
    
config = AutoConfig.from_pretrained(checkpoint)
model = BertForPairwiseCLS.from_pretrained(checkpoint, config = config).to(device)
print(model)

Using mps device


Some weights of BertForPairwiseCLS were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForPairwiseCLS(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
from tqdm.auto import tqdm  # tqdm库显示进度条，auto会自动选择适合当前环境的进度条版本（jupyter notebook或终端）
# 在训练循环中计算损失、优化模型的参数，在验证/测试循环中评估模型的性能
def train_loop(dataloader, model, loss_fn, optimizer, lr_scheduler, epoch, total_loss): # lr_scheduler：学习率调度器
    progress_bar = tqdm(range(len(dataloader)))  # 进度条长度等于数据批次数（step数）
    progress_bar.set_description(f'loss: {0:>7f}')  # 设置进度条的初始描述文字：右对齐显示损失值，宽度7，浮点数格式，初始显示 loss: 0.000000
    finish_step_num = (epoch-1)*len(dataloader)  # 计算前面的epoch已完成的总步数（批次数），用于计算全局平均损失，epoch应该是从1开始计数的
    
    model.train()
    for step, (X, y) in enumerate(dataloader, start=1):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()   # 反向传播，计算梯度
        optimizer.step()  # 更新模型参数（用计算出的梯度进行优化）
        lr_scheduler.step()  # 更新学习率，此处是每个step更新一次（通常每个step或每个epoch更新一次）

        total_loss += loss.item()  #loss.item() 将tensor转换为python数值（标量），记得加item
        progress_bar.set_description(f'loss: {total_loss/(finish_step_num + step):>7f}') # 更新进度条描述，计算平均损失：总损失 / 已完成步数；显示当前的平均损失值
        progress_bar.update(1) # 将进度条前进一步（刷新显示）
    return total_loss

In [ ]:
from tqdm.auto import tqdm

def train_loop(dataloader, model, loss_fn, optimizer, lr_schedule, epoch, total_loss):
    progress_bar = tqdm(range(len(dataloader)))
    progress_bar.set_description(f"loss:{0:>7f}")
    finish_step_num = (epoch - 1) * len(dataloader)

    model.train()
    for step, (X, y) in enumerate(dataloader, start = 1):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_schedule.step()

        total_loss += loss.item()

        progress_bar.set_description(f"avg loss: {total_loss/(finish_step_num + step):>7f}")
        progress_bar.update(1)
    return total_loss

In [ ]:
import torch.nn as nn
from torch.optim import AdamW
from transformers import get_scheduler

learning_rate = 1e-5
epoch_num = 3

loss_fn = nn.CrossEntropyLoss()

# 是model.parameters()，不是model.parameter()
optimizer = AdamW(model.parameters(), lr=learning_rate) # AdamW优化器将权重衰减解耦，直接作用于权重更新步骤

lr_scheduler = get_scheduler(
    "linear",  # 线性衰减学习率，从初始值开始，均匀地降到0
    optimizer=optimizer,
    num_warmup_steps=0,  # 预热步数为0。如果为100，那么在前100步，学习率会从0慢慢增加到1e-5
    num_training_steps=epoch_num*len(train_dataloader),
)

total_loss = 0.
for t in range(epoch_num):
    print(f"Epoch {t+1}/{epoch_num}\n-------------------------------")
    total_loss = train_loop(train_dataloader, model, loss_fn, optimizer, lr_scheduler, t+1, total_loss)
    test_loop(valid_dataloader, model, mode='Valid')
print("Done!")

In [21]:
import torch.nn as nn
from transformers import get_scheduler
from torch.optim import AdamW

learning_rate = 1e-5
epoch = 3

optimizer = AdamW(params = model.parameters(), lr = learning_rate)

lr_scheduler = get_scheduler(
    name = "linear",
    optimizer = optimizer,
    num_warmup_steps = 0,
    num_training_steps = epoch * len(train_dataloader)
)

In [ ]:
def test_loop(dataloader, model, mode='Test'):
    assert mode in ['Valid', 'Test']
    size = len(dataloader.dataset)  # 此处应该是dataset的大小（总的数据量），而不是dataloader的大小（除以batch_size之后的值）
    correct = 0

    model.eval()
    with torch.no_grad():
        for X, y in dataloader: # 这块如果用enumerate(dataloader)，返回的是（batch_index, (X, y)）,不加enumerate，返回的才是(X, y)
            X, y = X.to(device), y.to(device)
            pred = model(X)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    correct /= size
    print(f"{mode} Accuracy: {(100*correct):>0.1f}%\n")   # 转换为百分比，保留1位小数，右对齐
    return correct

total_loss = 0.
best_acc = 0.
for t in range(epoch_num):
    print(f"Epoch {t+1}/{epoch_num}\n-------------------------------")
    total_loss = train_loop(train_dataloader, model, loss_fn, optimizer, lr_scheduler, t+1, total_loss)
    valid_acc = test_loop(valid_dataloader, model, mode='Valid')
    if valid_acc > best_acc:
        best_acc = valid_acc
        print('saving new weights...\n')
        torch.save(model.state_dict(), f'epoch_{t+1}_valid_acc_{(100*valid_acc):0.1f}_model_weights.bin')
print("Done!")

In [28]:
def test_loop(dataloader, model, mode = "test"):
    assert mode in ["valid", "test"]

    correct = 0
    size = len(dataloader.dataset)

    model.eval()
    with torch.no_grad():
        # for X,y in enumerate(dataloader):   
        for X,y in dataloader:            
            X,y = X.to(device), y.to(device)
            pred = model(X)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    accuracy = correct / size

    print(f"{mode} accuracy: {(100 * accuracy):>0.1f}% \n")

    return accuracy

In [ ]:
train_loss = 0
best_acc = 0
loss_fn = nn.CrossEntropyLoss()

for ep in range(epoch):
    print(f"epoch:{ep + 1}==========\n")
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer, lr_scheduler, ep+1, train_loss)
    accuracy = test_loop(valid_dataloader, model, "valid")
    if accuracy > best_acc:
        best_acc = accuracy
        print("savint new weights====\n")
        torch.save(model.state_dict(), f"epoch_{ep+1}_valid_acc:{100*accuracy:0.1f}_model_weights.bin")

print("Done!")

epoch:1==========



  0%|          | 0/8584 [00:00<?, ?it/s]

valid accuracy: 81.0% 

savint new weights====

epoch:2==========



  0%|          | 0/8584 [00:00<?, ?it/s]

valid accuracy: 81.0% 

epoch:3==========



  0%|          | 0/8584 [00:00<?, ?it/s]

KeyboardInterrupt: 